# AI-sfa P1·P2 — Kaggle GPU

**처음 한 번만 설정(오른쪽 패널 또는 메뉴):**
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On**(휴대폰 인증한 계정만 켤 수 있다)
3. **Add-ons → Secrets**에 `RCLONE_GDRIVE_AUTH_B64`를 추가하고(값은 Claude 클라우드 환경 변수에 넣은 것과 같은 값) 이 노트북에 체크해 붙인다. 대화에는 붙이지 않는다.

**실행:** 오른쪽 위 **Save Version → Save & Run All (Commit)**을 누르면 브라우저를 닫아도 최대 12시간 돈다.
결과는 영상마다 Drive에 바로 올라간다(P2 → `02_분석자료/P2_카메라/`, P1 → `02_분석자료/포즈_RTMW/`).
12시간이 지나 멈추면 다시 Save & Run All을 누르면 끝난 영상은 건너뛴다. Kaggle GPU는 주 30시간까지 무료.

In [ ]:
# 1. 준비: rclone 설치·설정(인증값은 출력하지 않는다), 라이브러리
import os, json, base64, subprocess, pathlib
from kaggle_secrets import UserSecretsClient
raw = UserSecretsClient().get_secret('RCLONE_GDRIVE_AUTH_B64').strip()
d = json.loads(base64.b64decode(raw + '=' * (-len(raw) % 4)))
p = pathlib.Path.home() / '.config/rclone'; p.mkdir(parents=True, exist_ok=True)
lines = ['[gdrive]', 'type = drive', 'scope = drive.readonly,drive.file']
if d.get('client_id'): lines.append('client_id = ' + d['client_id'])
if d.get('client_secret'): lines.append('client_secret = ' + d['client_secret'])
lines.append('token = ' + d['token'])
(p / 'rclone.conf').write_text('\n'.join(lines) + '\n'); os.chmod(p / 'rclone.conf', 0o600)
del raw, d, lines
!cd /tmp && curl -sSLo rc.zip https://downloads.rclone.org/rclone-current-linux-amd64.zip && unzip -qo rc.zip && cp rclone-*/rclone /usr/local/bin/
!rclone lsd gdrive:AI-sfa 2>/dev/null | head -3
!pip -q install rtmlib onnxruntime-gpu onnx onnx2torch
import torch; print('GPU 수:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2. 작업자 파일 쓰기
open('/kaggle/working/kaggle_worker.py', 'w', encoding='utf-8').write("\"\"\"Kaggle GPU 작업자: P1(RTMW-x 포즈 전체 추출) 또는 P2(카메라 광류)를 영상마다 Drive에서 받아 처리하고 Drive에 올린다.\n\n사용: CUDA_VISIBLE_DEVICES=<gpu> python kaggle_worker.py <p1|p2> <작업자 번호> <작업자 수>\n- rclone 원격 이름은 gdrive(노트북이 설정). 끝난 영상(Drive에 결과 있음)은 건너뛴다.\n- 결과 형식은 Colab 노트북(tools/colab_p1_full_build.py, tools/colab_p2_build.py)과 같다.\n\"\"\"\nimport gzip\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nimport cv2\nimport numpy as np\n\nTASK, WID, NW = sys.argv[1], int(sys.argv[2]), int(sys.argv[3])\nSRC = \"gdrive:AI-sfa/01_원본영상\"\nDIRS = [\"yutaka_nakamura_디지털\", \"nian41\", \"riooo\", \"yukina_kosaka\", \"optical-core\"]  # 몸 학습 우선순위 순\nOUT = {\"p1\": \"gdrive:AI-sfa/02_분석자료/포즈_RTMW\", \"p2\": \"gdrive:AI-sfa/02_분석자료/P2_카메라\"}[TASK]\nEXT = {\"p1\": \".jsonl.gz\", \"p2\": \".npz\"}[TASK]\nTMP = f\"/tmp/w_{TASK}_{WID}\"\nos.makedirs(TMP, exist_ok=True)\n\n\ndef rc(*a):\n    return subprocess.run([\"rclone\", *a], capture_output=True, text=True)\n\n\ndef log(*a):\n    print(time.strftime(\"%H:%M:%S\"), TASK, WID, *a, flush=True)\n\n\nimport onnxruntime as ort  # noqa: E402\n\ntry:\n    ort.preload_dlls()  # torch에 든 CUDA·cuDNN을 먼저 불러 ONNX가 GPU를 쓰게 한다\nexcept Exception as e:\n    log(\"preload_dlls 실패\", e)\nimport torch  # noqa: E402\nfrom rtmlib import Wholebody  # noqa: E402\n\nODEV = \"cuda\" if \"CUDAExecutionProvider\" in ort.get_available_providers() else \"cpu\"\nWB = Wholebody(mode=\"performance\", backend=\"onnxruntime\", device=ODEV)\nWB.det_model.score_thr = 0.3\n\n\nclass TorchSession:\n    \"\"\"rtmlib이 쓰는 ONNX 세션 자리(get_inputs·get_outputs·run)에 GPU PyTorch 모델을 끼운다.\n    ONNX → 기본 최적화(상수 접기) → Clip의 빈 선택 입력 제거 → onnx2torch 변환. 서버에서 출력 차이 4e-6 이하 확인.\"\"\"\n\n    def __init__(self, sess, path):\n        import onnx\n        from onnx2torch import convert\n        so = ort.SessionOptions()\n        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_BASIC\n        so.optimized_model_filepath = path + \".opt.onnx\"\n        ort.InferenceSession(path, so, providers=[\"CPUExecutionProvider\"])\n        m = onnx.load(so.optimized_model_filepath)\n        for node in m.graph.node:\n            if node.op_type == \"Clip\":\n                while len(node.input) > 1 and node.input[-1] == \"\":\n                    node.input.pop()\n        self.m = convert(m).cuda().eval()\n        self.sess = sess\n\n    def get_inputs(self):\n        return self.sess.get_inputs()\n\n    def get_outputs(self):\n        return self.sess.get_outputs()\n\n    def get_providers(self):\n        return [\"TorchCUDA\"]\n\n    def run(self, names, feed):\n        with torch.no_grad():\n            out = self.m(torch.from_numpy(next(iter(feed.values()))).cuda())\n        out = out if isinstance(out, (list, tuple)) else [out]\n        return [o.float().cpu().numpy() for o in out]\n\n\nif ODEV == \"cpu\" and torch.cuda.is_available():\n    import glob as _g\n    for tool, key in ((WB.det_model, \"yolox\"), (WB.pose_model, \"rtmw\")):\n        path = [f for f in _g.glob(os.path.expanduser(\"~/.cache/rtmlib/hub/checkpoints/*.onnx\"))\n                if os.path.basename(f).startswith(key) and not f.endswith(\".opt.onnx\")][0]\n        tool.session = TorchSession(tool.session, path)\ntry:\n    log(\"ONNX 장치\", WB.det_model.session.get_providers()[0], WB.pose_model.session.get_providers()[0],\n        \"torch\", torch.cuda.get_device_name(0))\nexcept Exception:\n    log(\"ONNX 장치\", ODEV)\n\n\ndef r(a, n):\n    return np.round(np.asarray(a, dtype=float), n)\n\n\n# ---------- P1: 모든 프레임 RTMW-x 133점. 검출 상자 없으면 빈 프레임(거짓 뼈대 방지) ----------\ndef run_p1(loc, dst):\n    cap = cv2.VideoCapture(loc)\n    n = det = 0\n    with gzip.open(dst, \"wt\", encoding=\"utf-8\") as f:\n        while True:\n            ok, im = cap.read()\n            if not ok:\n                break\n            b = np.asarray(WB.det_model(im)).reshape(-1, 4)\n            ppl = []\n            if len(b):\n                kp, sc = WB.pose_model(im, bboxes=b)\n                for bb, k, s in zip(b, kp, sc):\n                    ppl.append({\"b\": r(bb, 1).tolist(), \"k\": np.concatenate([r(k, 1), r(s, 3)[:, None]], 1).tolist()})\n                det += 1\n            f.write(json.dumps({\"f\": n, \"p\": ppl}, separators=(\",\", \":\")) + \"\\n\")\n            n += 1\n    cap.release()\n    return f\"{n} 프레임 사람 {det}\"\n\n\n# ---------- P2: RAFT-large 광류 + 캐릭터 가림 + 닮음 변환 RANSAC ----------\nif TASK == \"p2\":\n    import torch.nn.functional as F\n    from torchvision.models.optical_flow import Raft_Large_Weights, raft_large\n\n    DEV = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    RAFT = raft_large(weights=Raft_Large_Weights.C_T_SKHT_V2).to(DEV).eval()\nH, DET_EVERY, UPDATES, B = 368, 6, 6, 16\n\n\ndef prep(ims, w):\n    t = torch.from_numpy(np.stack([cv2.cvtColor(cv2.resize(i, (w, H)), cv2.COLOR_BGR2RGB) for i in ims]))\n    return (t.permute(0, 3, 1, 2).float() / 127.5 - 1).to(DEV)\n\n\ndef fit(flow, mask):\n    hh, ww = flow.shape[:2]\n    ys, xs = np.mgrid[4:hh:8, 4:ww:8]\n    m = mask[ys, xs]\n    if m.sum() < 30:\n        return [0, 0, 0, 0, 0, 0]\n    p0 = np.stack([xs[m], ys[m]], 1).astype(np.float32)\n    p1 = p0 + flow[ys[m], xs[m]]\n    A, inl = cv2.estimateAffinePartial2D(p0, p1, method=cv2.RANSAC, ransacReprojThreshold=1.5, maxIters=2000)\n    med = float(np.median(np.linalg.norm(p1 - p0, axis=1)))\n    if A is None:\n        return [0, 0, 0, 0, 0, med]\n    s = np.hypot(A[0, 0], A[1, 0])\n    rot = np.arctan2(A[1, 0], A[0, 0])\n    c = np.array([ww / 2, hh / 2])\n    t = A[:, :2] @ c + A[:, 2] - c\n    return [t[0] / ww, t[1] / ww, float(np.log(s)), float(rot), float(inl.mean()), med]\n\n\ndef run_p2(loc, dst):\n    cap = cv2.VideoCapture(loc)\n    fps = cap.get(cv2.CAP_PROP_FPS)\n    ok, prev = cap.read()\n    if not ok:\n        return \"빈 영상\"\n    h0, w0 = prev.shape[:2]\n    w = int(round(w0 * H / h0 / 8)) * 8\n    last = [None]\n\n    def mask(im, i):\n        if i % DET_EVERY == 0 or last[0] is None:\n            b = np.asarray(WB.det_model(im)).reshape(-1, 4)\n            c = (b[:, :2] + b[:, 2:]) / 2\n            hw = (b[:, 2:] - b[:, :2]) * 0.55\n            last[0] = np.concatenate([c - hw, c + hw], 1)\n        mk = np.ones((H, w), bool)\n        for x1, y1, x2, y2 in last[0] * [w / w0, H / h0, w / w0, H / h0]:\n            mk[max(0, int(y1)):int(y2) + 1, max(0, int(x1)):int(x2) + 1] = False\n        return mk, len(last[0])\n\n    small = lambda im: cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (160, 90),  # noqa: E731\n                                  interpolation=cv2.INTER_AREA).astype(np.int16)\n    pm, n0 = mask(prev, 0)\n    nb, fi, nstill = [n0], 1, 0\n    cam = [[0] * 6]\n    grid = [np.zeros((9, 16, 2), np.float16)]\n    done = False\n    while not done:\n        cur = []\n        while len(cur) < B:\n            ok, im = cap.read()\n            if not ok:\n                done = True\n                break\n            cur.append(im)\n        if not cur:\n            break\n        ms = [mask(im, fi + k) for k, im in enumerate(cur)]\n        fi += len(cur)\n        pairs = [prev] + cur\n        # 애니의 정지 프레임(같은 그림 유지)은 광류를 계산하지 않고 카메라 정지로 기록한다\n        moving = [k for k in range(len(cur)) if np.abs(small(pairs[k]) - small(pairs[k + 1])).mean() > 0.6]\n        if moving:\n            a = prep([pairs[k] for k in moving], w)\n            b = prep([pairs[k + 1] for k in moving], w)\n            with torch.no_grad():\n                fl = RAFT(a, b, num_flow_updates=UPDATES)[-1]\n            g = F.adaptive_avg_pool2d(fl, (9, 16)).permute(0, 2, 3, 1).cpu().numpy() / w\n            fl = fl.permute(0, 2, 3, 1).cpu().numpy()\n        masks = [pm] + [m for m, _ in ms]\n        pos = {k: j for j, k in enumerate(moving)}\n        for k in range(len(cur)):\n            if k in pos:\n                cam.append(fit(fl[pos[k]], masks[k] & masks[k + 1]))\n                grid.append(g[pos[k]].astype(np.float16))\n            else:\n                nstill += 1\n                cam.append([0, 0, 0, 0, 1.0, 0])\n                grid.append(np.zeros((9, 16, 2), np.float16))\n        nb += [n for _, n in ms]\n        prev = cur[-1]\n        pm = ms[-1][0]\n    cap.release()\n    with open(dst, \"wb\") as fh:\n        np.savez_compressed(fh, cam=np.array(cam, np.float32), grid=np.stack(grid), boxes=np.array(nb), fps=fps,\n                            size=[w0, h0])\n    return f\"{len(cam)} 프레임(정지 {nstill})\"\n\n\ndef main():\n    vids = []\n    for d in DIRS:\n        an = \"yutaka_nakamura\" if d.startswith(\"yutaka\") else d\n        done = set(rc(\"lsf\", f\"{OUT}/{an}\").stdout.split())\n        for rel in sorted(rc(\"lsf\", \"-R\", \"--files-only\", \"--include\", \"*.mp4\", f\"{SRC}/{d}\").stdout.splitlines()):\n            pid = os.path.basename(rel).split(\"_\")[0].split(\".\")[0]\n            if pid + EXT not in done:\n                vids.append((f\"{SRC}/{d}/{rel}\", f\"{OUT}/{an}/{pid}{EXT}\"))\n    mine = vids[WID::NW]\n    log(\"남은 영상\", len(vids), \"내 몫\", len(mine))\n    t0 = time.time()\n    for k, (src, dst) in enumerate(mine):\n        t = time.time()\n        loc = f\"{TMP}/cur.mp4\"\n        out = f\"{TMP}/out{EXT}\"\n        try:\n            rc(\"copyto\", src, loc)\n            msg = (run_p1 if TASK == \"p1\" else run_p2)(loc, out)\n            up = rc(\"copyto\", out, dst)\n            if up.returncode:\n                log(\"올리기 실패\", dst, up.stderr[-300:])\n                continue\n        except Exception:\n            traceback.print_exc()\n            continue\n        log(k + 1, \"/\", len(mine), os.path.basename(dst), msg, round(time.time() - t), \"초\", \"누적\",\n            round((time.time() - t0) / 60), \"분\")\n    log(\"끝\")\n\n\nif __name__ == \"__main__\":\n    main()\n")
print('작업자 준비')

In [ ]:
# 3. 실행: GPU 0 = P2(카메라), GPU 1 = P1(포즈). 5분마다 진행 상황을 찍는다
import subprocess, time, os
ng = torch.cuda.device_count()
def start(task, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    return subprocess.Popen(['python', '/kaggle/working/kaggle_worker.py', task, '0', '1'], env=env,
                            stdout=open(f'/kaggle/working/{task}.log', 'w'), stderr=subprocess.STDOUT)
# 두 작업자가 같은 모델 파일을 동시에 받다 충돌하지 않도록 먼저 한 번 받아 둔다
subprocess.run(['python', '-c', "from rtmlib import Wholebody; Wholebody(mode='performance', backend='onnxruntime', device='cpu')"], check=True)
plan = [('p2', 0), ('p1', 1)] if ng >= 2 else [('p2', 0)]
procs = {t: start(t, g) for t, g in plan}
t0 = time.time()
while procs and time.time() - t0 < 11.6 * 3600:
    time.sleep(300)
    for t, pr in list(procs.items()):
        tail = open(f'/kaggle/working/{t}.log', encoding='utf-8', errors='replace').read().splitlines()[-2:]
        print(round((time.time() - t0) / 60), '분', t, '|', ' / '.join(tail), flush=True)
        if pr.poll() is not None:
            del procs[t]
            if ng < 2 and t == 'p2': procs['p1'] = start('p1', 0)
for pr in procs.values(): pr.terminate()
print('종료')